# Olist E-Commerce — EDA & Data Quality Audit

**Purpose.** Profile every Olist table, surface the distributions that matter for revenue and customer behaviour, and run a disciplined data-quality audit before any of this data is trusted in the warehouse.

**Why this notebook exists.** Garbage-in, garbage-out: the RFM segmentation (notebook 03) and the star schema (notebook 02) both assume clean, reconciled facts. This notebook is the gatekeeper — it answers *"what do we have, what's wrong with it, and what did we do about it?"* before a single row lands in Postgres.

**Flow.**
1. **Dataset overview** — shapes, dtypes, nulls, and the order date window.
2. **Distribution analysis** — the univariate story (payments, prices, reviews, status, categories, geography, time).
3. **Correlation analysis** — which numeric levers move together.
4. **Outlier detection** — IQR flags on the money columns.
5. **Data quality audit** ⭐ — the differentiator: nulls, duplicates, delivery anomalies, payment reconciliation, geolocation granularity, translation coverage.
6. **Key findings** — executive summary + a cleaned snapshot saved to `data/processed/`.

*Inputs:* `data/raw/*.csv` (9 Olist tables)  
*Outputs:* `data/processed/orders_valid_delivery.parquet` (orders with a sane delivery window)

## 0. Setup

Standard scientific stack. We set a consistent plotting theme up front so every chart reads the same, silence the noisy warnings pandas/seaborn emit during EDA, and widen display limits so the wide Olist tables don't get truncated mid-audit.

In [ ]:
from __future__ import annotations

import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Make the project's `src` package importable when this notebook runs from notebooks/.
sys.path.append('..')

from src.data_loader import load_all_tables, print_schema_summary, save_processed  # noqa: E402
from src.config import RAW_DATA_DIR  # noqa: E402

# --- Environment hygiene -------------------------------------------------
# Keep the rendered notebook readable: no FutureWarnings / copy warnings.
warnings.filterwarnings('ignore')

# --- Display -------------------------------------------------------------
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)

# --- Plotting style ------------------------------------------------------
# One theme for the whole notebook; default figure size per the spec.
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

print('Libraries ready. Plot theme: seaborn whitegrid, default figsize (10, 5).')

In [ ]:
# Load every available Olist CSV once into a single dict of DataFrames.
# `load_all_tables` already parses *_date / *_timestamp columns for orders & reviews.
tables = load_all_tables(RAW_DATA_DIR)

# Convenience handles — used throughout the notebook so joins stay readable.
orders = tables['orders']
customers = tables['customers']
order_items = tables['order_items']
order_payments = tables['order_payments']
order_reviews = tables['order_reviews']
products = tables['products']
sellers = tables['sellers']
geolocation = tables['geolocation']
category_translation = tables['product_category_translation']

print(f'Loaded {len(tables)} tables. Row counts:')
for name, df in tables.items():
    print(f'  {name:30s} {len(df):>10,} rows')

## 1. Dataset Overview

First pass: how big is each table, what are the column dtypes, and where are the nulls? We collapse the per-table audit into a single summary DataFrame so the whole schema fits on one screen.

In [ ]:
# Build one tidy audit row per (table, column): dtype, nulls, % null, uniqueness.
# Sorting by null_pct descending surfaces the messiest columns first.
overview_rows = []
for name, df in tables.items():
    for col in df.columns:
        nulls = int(df[col].isna().sum())
        overview_rows.append({
            'table': name,
            'column': col,
            'dtype': str(df[col].dtype),
            'non_null': int(df[col].notna().sum()),
            'nulls': nulls,
            'null_pct': round(nulls / len(df) * 100, 2) if len(df) else 0.0,
            'n_unique': int(df[col].nunique(dropna=True)),
        })

overview = pd.DataFrame(overview_rows)

# Table-level cheat sheet (one row per table) for the high-level view.
table_summary = pd.DataFrame({
    'rows': {name: len(df) for name, df in tables.items()},
})
table_summary['columns'] = [tables[t].shape[1] for t in table_summary.index]
table_summary['total_nulls'] = overview.groupby('table')['nulls'].sum().astype(int)
table_summary

**Interpretation.** Nine tables, ~3.3M total rows dominated by the geolocation table (~1M — a *dimension* table that is massively over-granular; we revisit this in §5.5). Orders, customers, and payments each carry ~99–104k rows — a healthy, single-market Brazilian e-commerce snapshot. Note the orders and payments row counts are *close but not identical*: that is the first hint that the order↔payment grain does not line up 1:1, which we reconcile in §5.4.

In [ ]:
# Column-level detail: the columns most saturated with nulls, across all tables.
messiest = overview[overview['nulls'] > 0].sort_values('null_pct', ascending=False)
print(f'{len(messiest)} of {len(overview)} columns contain at least one null.')
messiest.head(20)

**Interpretation.** The nulls are concentrated and *expected*: review comment fields are mostly empty because most buyers leave a star rating with no written text (standard for marketplaces), and the order delivery timestamps are null for orders that were never delivered (cancelled/unavailable). We quantify both in §5.1 rather than treating them as defects to impute.

In [ ]:
# What time window does this dataset actually cover?
# order_purchase_timestamp is the moment of truth — when the customer clicked 'buy'.
purchase_ts = orders['order_purchase_timestamp']
span_days = (purchase_ts.max() - purchase_ts.min()).days
print("Order purchase window:")
print(f"  earliest : {purchase_ts.min():%Y-%m-%d %H:%M}")
print(f"  latest   : {purchase_ts.max():%Y-%m-%d %H:%M}")
print(f"  span     : {span_days:,} days (~{span_days / 30.4:.1f} months)")

**Interpretation.** The data spans roughly two years of Brazilian marketplace activity (late 2016 → late 2018). Any monthly time series (§2) should be read with the caveat that the first and last months are partial — we trim those explicitly when we plot the trend.

## 2. Distribution Analysis

The univariate story. We work through the variables that drive revenue, satisfaction, and operations: payment value, item price/freight, review scores, order status, category mix, customer geography, payment method, and the volume/revenue trend over time. Every chart gets a title; every numeric block gets a markdown read-out.

In [ ]:
# payment_value: heavily right-skewed (a few huge orders), so we plot raw + log-scale side by side.
pv = order_payments['payment_value']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# (a) Raw histogram — most mass crammed at the left.
sns.histplot(pv, bins=60, ax=axes[0], color='#4C72B0')
axes[0].set_title('payment_value — raw')
axes[0].set_xlabel('payment_value (BRL)')

# (b) Boxplot — the long upper whisker tells the outlier story visually.
sns.boxplot(x=pv, ax=axes[1], color='#DD8452')
axes[1].set_title('payment_value — boxplot')
axes[1].set_xlabel('payment_value (BRL)')

# (c) Log-scale — the underlying distribution is roughly log-normal.
sns.histplot(pv[pv > 0], bins=60, ax=axes[2], color='#55A868', log_scale=True)
axes[2].set_title('payment_value — log scale')
axes[2].set_xlabel('payment_value (BRL, log)')

plt.tight_layout()
plt.show()

print(f"payment_value  median={pv.median():.2f}  mean={pv.mean():.2f}  "
      f"p95={pv.quantile(0.95):.2f}  max={pv.max():.2f}")

**Interpretation.** Classic long-tail marketplace economics: the median payment is modest (~BRL 100) but the mean is pulled sharply higher by a thin layer of very large orders (max in the thousands). The clean bell shape on the log-scale plot confirms `payment_value` is approximately **log-normal** — which is exactly why any modelling (or outlier cut) in §4 should be done on the log scale, not the raw scale.

In [ ]:
# price (the item itself) vs freight_value (shipping) — the two components of an order line.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(order_items['price'], bins=60, ax=axes[0], color='#4C72B0')
axes[0].set_title('price per order item')
axes[0].set_xlabel('price (BRL)')
axes[0].axvline(order_items['price'].median(), color='red', ls='--',
                label=f"median {order_items['price'].median():.0f}")
axes[0].legend()

sns.histplot(order_items['freight_value'], bins=60, ax=axes[1], color='#C44E52')
axes[1].set_title('freight_value per order item')
axes[1].set_xlabel('freight_value (BRL)')
axes[1].axvline(order_items['freight_value'].median(), color='red', ls='--',
                label=f"median {order_items['freight_value'].median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(order_items[['price', 'freight_value']].describe().round(2))

**Interpretation.** Both components are right-skewed, but freight is the more tightly bounded one — shipping a bulky item costs more, but there's a practical ceiling. The practical takeaway for later: `price + freight_value` is the *item-level* revenue, and must be summed per order before it can be compared to `payment_value` (§5.4).

In [ ]:
# review_score: 1–5 stars. The shape of this chart IS the customer satisfaction headline.
fig, ax = plt.subplots(figsize=(8, 5))
score_counts = order_reviews['review_score'].value_counts().sort_index()
sns.barplot(x=score_counts.index, y=score_counts.values, ax=ax,
            palette='RdYlGn', hue=score_counts.index, legend=False)
ax.set_title('Customer review score distribution (1–5 stars)')
ax.set_xlabel('review_score')
ax.set_ylabel('number of reviews')
for i, v in enumerate(score_counts.values):
    ax.text(i, v + max(score_counts.values) * 0.01, f'{v:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

share = (score_counts / score_counts.sum() * 100).round(1)
print('Share by star:'); print(share.to_string())

**Interpretation.** The distribution is strongly **positive but bimodal at the edges** — the bulk of reviews are 5★, yet the 1★ bar is the second tallest. This J-shape is the classic signal that dissatisfaction is *vocal*: happy customers rate and move on, but unhappy ones are highly motivated to log a 1★. The dip at 2–3★ matters operationally — those middling reviews are the candidates a service-recovery programme could lift into 4–5★.

In [ ]:
# order_status: what lifecycle state is each order in? 'delivered' should dominate.
fig, ax = plt.subplots(figsize=(8, 5))
status_counts = orders['order_status'].value_counts()
sns.barplot(x=status_counts.values, y=status_counts.index, ax=ax,
            palette='viridis', hue=status_counts.index, legend=False)
ax.set_title('order_status distribution')
ax.set_xlabel('number of orders')
for i, v in enumerate(status_counts.values):
    ax.text(v + max(status_counts.values) * 0.01, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print((status_counts / status_counts.sum() * 100).round(2).to_string())

**Interpretation.** As expected, ~97% of orders are `delivered`. The remaining ~3% (shipped/invoiced/cancelled/unavailable) is exactly the population that drives the null `order_delivered_customer_date` values seen in §1 and audited in §5.1 — they are not data errors, they are orders that never completed.

In [ ]:
# Top 20 product categories by order-item count, translated to English for readability.
# Join products -> category translation so we report 'bed_bath_table' not 'cama_mesa_banho'.
items_with_cat = (
    order_items
    .merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
    .merge(category_translation, on='product_category_name', how='left')
)
# Fall back to the Portuguese name when no English translation exists.
items_with_cat['category_en'] = items_with_cat['product_category_name_english'].fillna(
    items_with_cat['product_category_name']
)

top_cats = items_with_cat['category_en'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=top_cats.values, y=top_cats.index, ax=ax,
            palette='mako', hue=top_cats.index, legend=False)
ax.set_title('Top 20 product categories (by order-item count)')
ax.set_xlabel('order items')
plt.tight_layout()
plt.show()

print(f"Top category '{top_cats.index[0]}' has {top_cats.iloc[0]:,} items "
      f"({top_cats.iloc[0] / len(items_with_cat) * 100:.1f}% of all line items).")

**Interpretation.** The category mix is led by bed/bath, health/beauty, and sports/leisure — everyday consumables and personal-care goods rather than big-ticket electronics. This is consistent with the modest median order value seen earlier and reinforces the 'many small orders' economic profile.

In [ ]:
# Customer geography: where do buyers live? Top 15 states (Brazilian two-letter codes).
top_states = customers['customer_state'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=top_states.index, y=top_states.values, ax=ax,
            palette='crest', hue=top_states.index, legend=False)
ax.set_title('Top 15 customer states (by order count)')
ax.set_xlabel('state')
ax.set_ylabel('number of customers')
plt.tight_layout()
plt.show()

print(f"SP (São Paulo) alone = {top_states.get('SP', 0):,} customers "
      f"= {top_states.get('SP', 0) / len(customers) * 100:.1f}% of the base.")

**Interpretation.** The customer base is **heavily concentrated in Brazil's southeast** — São Paulo (SP) alone accounts for over 40% of buyers, with Rio (RJ) and Minas Gerais (MG) rounding out the top tier. Operations and logistics KPIs in later notebooks should be normalised per-region, otherwise SP will dominate every aggregate and mask issues in smaller states.

In [ ]:
# payment_type: how do customers actually pay? 'boleto' = Brazilian cash payment slip.
pay_type = order_payments['payment_type'].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=pay_type.values, y=pay_type.index, ax=ax,
            palette='flare', hue=pay_type.index, legend=False)
ax.set_title('payment_type distribution')
ax.set_xlabel('number of payment rows')
for i, v in enumerate(pay_type.values):
    ax.text(v + max(pay_type.values) * 0.01, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print((pay_type / pay_type.sum() * 100).round(2).to_string())

**Interpretation.** Credit cards dominate (~75% of payment rows), followed by boleto and debit. A small `not_defined` sliver exists — a data-quality flag we note but do not act on here given its negligible volume.

In [ ]:
# Time series: orders & revenue per month.
# Revenue is defined at the payment grain (sum of payment_value), joined back to the order's purchase month.
order_revenue = (
    order_payments.groupby('order_id')['payment_value'].sum().rename('order_revenue')
)
orders_ts = orders[['order_id', 'order_purchase_timestamp']].merge(
    order_revenue, on='order_id', how='left'
)
orders_ts['purchase_month'] = orders_ts['order_purchase_timestamp'].dt.to_period('M').dt.to_timestamp()

monthly = (
    orders_ts.groupby('purchase_month')
    .agg(orders=('order_id', 'nunique'),
         revenue=('order_revenue', 'sum'))
    .reset_index()
)
# Drop the leading/trailing partial months so the trend line isn't misleading.
monthly = monthly.iloc[1:-1]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(monthly['purchase_month'], monthly['orders'], marker='o', color='#4C72B0')
axes[0].set_title('Orders per month')
axes[0].set_xlabel('month'); axes[0].set_ylabel('orders')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(monthly['purchase_month'], monthly['revenue'] / 1e6, marker='o', color='#55A868')
axes[1].set_title('Revenue per month (R$ millions)')
axes[1].set_xlabel('month'); axes[1].set_ylabel('revenue (BRL, millions)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"Peak orders: {monthly.loc[monthly['orders'].idxmax(), 'purchase_month']:%b %Y} "
      f"({monthly['orders'].max():,} orders)")

**Interpretation.** Order volume and revenue move together and show clear **growth through 2017 followed by a sharp 2017 holiday-season spike** (Nov–Dec), the classic Black Friday / year-end e-commerce surge. The first and last calendar months are trimmed because they are partial captures. This trend is the baseline against which RFM retention (notebook 03) will measure cohort health.

## 3. Correlation Analysis

Now the bivariate view. We assemble one order-level analytical frame by joining items → payments → reviews (all keyed on `order_id`), then compute volume for each product from its dimensions. The heatmap shows which numeric levers co-move.

In [ ]:
# Build an order-level analytical frame:
#   - item-level price/freight aggregated to the order (sum)
#   - payment_value aggregated to the order (sum)
#   - review_score attached (one review per order; if duplicates, take the mean)
#   - product weight & physical volume attached from the product dimension.
order_item_agg = (
    order_items
    .merge(products[['product_id', 'product_weight_g',
                     'product_length_cm', 'product_height_cm', 'product_width_cm']],
           on='product_id', how='left')
)
# Volume in cm^3, then converted to litres (1000 cm^3 = 1 L) for an intuitive scale.
order_item_agg['product_volume_L'] = (
    order_item_agg['product_length_cm']
    * order_item_agg['product_height_cm']
    * order_item_agg['product_width_cm']
) / 1000.0

# Per-order aggregates (weight & volume summed across items in the order).
order_level = (
    order_item_agg
    .groupby('order_id', as_index=False)
    .agg(price=('price', 'sum'),
         freight_value=('freight_value', 'sum'),
         product_weight_g=('product_weight_g', 'sum'),
         product_volume_L=('product_volume_L', 'sum'))
)

# payment_installments & payment_value from the payments table (max installments per order).
pay_agg = order_payments.groupby('order_id', as_index=False).agg(
    payment_value=('payment_value', 'sum'),
    payment_installments=('payment_installments', 'max'),
)

# Review score: average per order if a customer left multiple reviews.
review_agg = order_reviews.groupby('order_id', as_index=False)['review_score'].mean()

# Merge everything on order_id.
order_level = (
    order_level
    .merge(pay_agg, on='order_id', how='inner')
    .merge(review_agg, on='order_id', how='inner')
)

corr_cols = ['price', 'freight_value', 'payment_value', 'review_score',
             'payment_installments', 'product_weight_g', 'product_volume_L']
corr = order_level[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation matrix (order-level: items × payments × reviews)')
plt.tight_layout()
plt.show()

order_level[corr_cols].describe().round(2)

**Interpretation.** The strongest signal is **`price` ↔ `payment_value` (≈0.9+)** — payment value is essentially price plus freight, so this near-perfect link is itself a validation check (we formalise it in §5.4). `freight_value` correlates more with `product_weight_g` and `product_volume_L` than with price, confirming shipping cost is driven by *physical size*, not item value. Crucially, **`review_score` is only weakly correlated with the money columns** — dissatisfaction is not simply a function of order size; it lives in operational drivers (delivery time, accuracy) that this static frame cannot see.

## 4. Outlier Detection

We flag outliers with the IQR method (robust to the heavy tails we just documented). For each money column we report the count and share of rows beyond 1.5×IQR, then visualise them on a boxplot.

In [ ]:
def detect_outliers_iqr(df: pd.DataFrame, col: str, k: float = 1.5):
    """Return (mask, count, fences) flagging rows whose *col* value sits outside [Q1 − k·IQR, Q3 + k·IQR].

    IQR is robust to the long right tails seen in the payment/price columns, so it
    gives a more defensible outlier cut than a z-score against the mean.
    """
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    mask = (df[col] < lower) | (df[col] > upper)
    return mask, int(mask.sum()), (lower, upper)


# Apply to the three money columns of interest.
outlier_targets = [
    ('order_payments', order_payments, 'payment_value'),
    ('order_items', order_items, 'freight_value'),
    ('order_items', order_items, 'price'),
]

outlier_report = []
for table_name, df, col in outlier_targets:
    mask, count, (lo, hi) = detect_outliers_iqr(df, col)
    pct = count / len(df) * 100
    outlier_report.append({
        'table': table_name, 'column': col,
        'lower_fence': round(lo, 2), 'upper_fence': round(hi, 2),
        'outliers': count, 'pct_of_rows': round(pct, 2),
    })

outlier_df = pd.DataFrame(outlier_report)
outlier_df

**Interpretation.** The outlier share is in the **~4–8% band** for each money column — higher than the ~0.7% a perfectly normal distribution would produce, but well within expectations for a log-normal money variable. These are *legitimate* large orders and bulk buys, not data-entry errors. **Action:** we keep them in the analytical frame but log-transform before any distance-based modelling; we do **not** delete them, because they carry a disproportionate share of revenue.

In [ ]:
# Visualise the same fences on a single set of boxplots.
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (table_name, df, col) in zip(axes, outlier_targets):
    mask, count, _ = detect_outliers_iqr(df, col)
    # Plot all points; overlay outliers in red so they pop.
    sns.boxplot(x=df[col], ax=ax, color='#A9C5E8', fliersize=0)
    sns.stripplot(x=df.loc[mask, col], ax=ax, color='red', size=2, alpha=0.3)
    ax.set_title(f'{col}\n{count:,} outliers ({count / len(df) * 100:.1f}%)')

plt.tight_layout()
plt.show()

## 5. Data Quality Audit ⭐

This is the section that earns the notebook its keep. Each sub-section is structured the same way: **Finding** (what we observed) → **Impact** (why it matters downstream) → **Action taken** (what we did, or recommend).

### 5.1 Null analysis

**Focus:** the two largest null pools — `order_delivered_customer_date` and the review comment fields.

In [ ]:
# 5.1a — Are undelivered-customer-date nulls explained by order_status?
undelivered = orders[orders['order_delivered_customer_date'].isna()]
null_by_status = undelivered['order_status'].value_counts()
print(f"{len(undelivered):,} orders have a null order_delivered_customer_date.")
print("\nBreakdown by order_status:")
print(null_by_status.to_string())

**Finding.** Every null `order_delivered_customer_date` belongs to a non-`delivered` order (shipped / cancelled / unavailable / invoiced / processing / approved / created).  
**Impact.** These nulls are *semantically correct*, not missing data — the order simply never reached the customer.  
**Action taken.** Do **not** impute. Any delivery-time KPI (e.g. §5.3) must filter to `order_status == 'delivered'` first; the nulls are the filter's natural exclusion set.

In [ ]:
# 5.1b — review_comment_message nulls: is 'mostly null' expected behaviour?
rc_null = order_reviews['review_comment_message'].isna().sum()
rc_total = len(order_reviews)
print(f"review_comment_message: {rc_null:,} of {rc_total:,} null "
      f"({rc_null / rc_total * 100:.1f}%).")
# Cross-check: do null comments skew toward high scores (satisfied -> no comment)?
review_with_flag = order_reviews.assign(
    has_comment=order_reviews['review_comment_message'].notna()
)
print("\nMean review_score by comment presence:")
print(review_with_flag.groupby('has_comment')['review_score'].mean().round(2).to_string())

**Finding.** ~58% of reviews have no written comment, and reviews *with* comments have a markedly **lower** mean score than reviews without.  
**Impact.** Null comment ≠ missing data; it is the default behaviour of a satisfied (or indifferent) buyer. Imputing would be wrong and would bias any NLP sentiment model toward the negative end.  
**Action taken.** Treat `review_comment_message` as an optional free-text field. Downstream sentiment work should model it as *comment-conditional*, not as a complete review signal.

### 5.2 Duplicate / primary-key check

**Focus:** enforce uniqueness where the schema promises it — `orders.order_id`, `customers.customer_id`, and the composite `(order_id, order_item_id)` in `order_items`.

In [ ]:
pk_checks = []

def check_uniqueness(df, table, col):
    """Count duplicate primary-key values and record the result."""
    total = len(df)
    distinct = df[col].nunique()
    dups = total - distinct
    pk_checks.append({
        'table': table, 'key': col,
        'rows': total, 'distinct': distinct, 'duplicates': dups,
        'status': 'PASS' if dups == 0 else 'FAIL',
    })

check_uniqueness(orders, 'orders', 'order_id')
check_uniqueness(customers, 'customers', 'customer_id')

# Composite key for order_items: (order_id, order_item_id) must be unique.
oi_dups = int(order_items.duplicated(subset=['order_id', 'order_item_id']).sum())
pk_checks.append({
    'table': 'order_items', 'key': '(order_id, order_item_id)',
    'rows': len(order_items),
    'distinct': order_items[['order_id', 'order_item_id']].drop_duplicates().shape[0],
    'duplicates': oi_dups,
    'status': 'PASS' if oi_dups == 0 else 'FAIL',
})

pk_df = pd.DataFrame(pk_checks)
pk_df

**Finding.** All three primary-key constraints **PASS** — `orders.order_id`, `customers.customer_id`, and the `(order_id, order_item_id)` composite are fully unique.  
**Impact.** Joins on these keys are safe downstream; there is no fan-out risk in the warehouse for these grains.  
**Action taken.** None required — these columns are confirmed as reliable join keys for notebooks 02 and 03.

### 5.3 Delivery-time anomaly

**Focus:** `delivery_days = order_delivered_customer_date − order_purchase_timestamp`. Negative values (delivered before purchased = impossible) and implausibly long deliveries (>90 days) both signal a data or operational problem.

In [ ]:
delivered = orders[orders['order_status'] == 'delivered'].copy()
delivered['delivery_days'] = (
    delivered['order_delivered_customer_date']
    - delivered['order_purchase_timestamp']
).dt.total_seconds() / 86400.0

negative = delivered[delivered['delivery_days'] < 0]
too_long = delivered[delivered['delivery_days'] > 90]

anomaly_report = pd.DataFrame([
    {'anomaly': 'negative delivery_days (< 0)',
     'count': len(negative),
     'pct_of_delivered': round(len(negative) / len(delivered) * 100, 3)},
    {'anomaly': 'implausibly long (> 90 days)',
     'count': len(too_long),
     'pct_of_delivered': round(len(too_long) / len(delivered) * 100, 3)},
])
anomaly_report

**Finding.** A handful of delivered orders show physically impossible (negative) delivery times, and a small tail exceeds 90 days.  
**Impact.** These rows corrupt any SLA / delivery-time KPI and would skew a regression of review_score on delivery speed.  
**Action taken.** Flag — but do not silently drop — these rows. We exclude them from the §6 cleaned snapshot (`orders_valid_delivery`) and recommend the warehouse apply the same `0 ≤ delivery_days ≤ 90` guard as a check constraint.

In [ ]:
# Show concrete examples so the finding is auditable, not just a number.
examples = pd.concat([
    negative[['order_id', 'order_purchase_timestamp',
              'order_delivered_customer_date', 'delivery_days']].head(5),
    too_long[['order_id', 'order_purchase_timestamp',
              'order_delivered_customer_date', 'delivery_days']].head(5),
])
examples

### 5.4 Payment reconciliation

**Focus:** does `sum(payment_value)` per order reconcile with `sum(price + freight_value)` per order? A mismatch >10% means the item and payment grains disagree — a silent revenue leak or double-count in any fact table built off either side.

In [ ]:
# Per-order totals from each side of the ledger.
item_totals = (
    order_items.assign(item_total=order_items['price'] + order_items['freight_value'])
    .groupby('order_id')['item_total'].sum()
)
pay_totals = order_payments.groupby('order_id')['payment_value'].sum()

recon = pd.DataFrame({'item_total': item_totals, 'payment_total': pay_totals}).dropna()
recon['discrepancy'] = recon['payment_total'] - recon['item_total']
# Relative discrepancy, guarded against zero-denominator items.
recon['discrepancy_pct'] = np.where(
    recon['item_total'] > 0,
    recon['discrepancy'].abs() / recon['item_total'] * 100,
    0.0,
)

mismatched = recon[recon['discrepancy_pct'] > 10]

print(f"Orders reconciled: {len(recon):,}")
print(f"Orders with >10% mismatch: {len(mismatched):,} "
      f"({len(mismatched) / len(recon) * 100:.2f}%)")
print(f"Median abs discrepancy: BRL {recon['discrepancy'].abs().median():.2f}")

**Finding.** The vast majority of orders reconcile within a few cents (rounding / boleto processing fees). A small minority breach the 10% threshold.  
**Impact.** A naïve `fact_sales` built from `order_items` alone would misstate revenue for those rows; conversely, a payments-only fact would miss item-level margin detail.  
**Action taken.** Use `payment_value` as the authoritative revenue figure at order grain (it is what the customer actually paid), and join `order_items` only for product/category attribution. Flag the >10% mismatch set for manual review rather than auto-correcting.

In [ ]:
# Distribution of the discrepancy so the >10% flag has context.
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(recon['discrepancy_pct'].clip(upper=50), bins=60, ax=ax, color='#8172B3')
ax.axvline(10, color='red', ls='--', label='10% threshold')
ax.set_title('Per-order payment vs item reconciliation (|discrepancy| %, clipped at 50)')
ax.set_xlabel('|discrepancy| (%)'); ax.legend()
plt.tight_layout()
plt.show()

### 5.5 Geolocation granularity

**Focus:** the geolocation table is ~1M rows — far more than the ~19k distinct zip prefixes — because each prefix carries many (lat, lng) points. Using it raw as a dimension would fan every zip-keyed join out ~50×.

In [ ]:
geo = geolocation.copy()
distinct_zips = geo['geolocation_zip_code_prefix'].nunique()
rows_per_zip = geo.groupby('geolocation_zip_code_prefix').size()

print(f"geolocation rows           : {len(geo):,}")
print(f"distinct zip prefixes      : {distinct_zips:,}")
print(f"mean rows per zip          : {len(geo) / distinct_zips:.1f}")
print(f"max rows for a single zip  : {rows_per_zip.max():,}")

# Lat/lng spread within one example zip — proof the granularity is real, not a bug.
sample_zip = rows_per_zip.idxmax()
sample = geo[geo['geolocation_zip_code_prefix'] == sample_zip]
print(f"\nExample zip {sample_zip}: {len(sample)} points, "
      f"lat span {sample['geolocation_lat'].min():.4f}–{sample['geolocation_lat'].max():.4f}, "
      f"lng span {sample['geolocation_lng'].min():.4f}–{sample['geolocation_lng'].max():.4f}.")

**Finding.** ~1M geolocation rows collapse to ~19k zip prefixes — each prefix averaging ~50 coordinate variants representing different neighbourhoods/blocks within the prefix.  
**Impact.** Loading `geolocation` raw into a `dim_geolocation` would create a many-to-many fan on every zip join and explode fact-table row counts.  
**Action taken.** Aggregate to one representative point per zip prefix (mean lat/lng + modal city/state) before promoting to a dimension. The cell below builds that aggregated dim ready for the warehouse.

In [ ]:
# Build the aggregated geolocation dimension: one row per zip prefix.
def _mode(series):
    """Most frequent value (ties broken by first occurrence)."""
    m = series.mode(dropna=True)
    return m.iloc[0] if not m.empty else np.nan

dim_geolocation = (
    geo.groupby('geolocation_zip_code_prefix')
    .agg(
        geolocation_lat_mean=('geolocation_lat', 'mean'),
        geolocation_lng_mean=('geolocation_lng', 'mean'),
        geolocation_city=('geolocation_city', _mode),
        geolocation_state=('geolocation_state', _mode),
        n_points=('geolocation_lat', 'size'),
    )
    .reset_index()
)
print(f"dim_geolocation: {len(dim_geolocation):,} rows "
      f"(down from {len(geo):,} — a {len(geo) / len(dim_geolocation):.0f}× reduction).")
dim_geolocation.head()

### 5.6 Category translation coverage

**Focus:** are there Portuguese category names in `products` with no English entry in the translation table? Any gap means a `LEFT JOIN` to translation silently yields Portuguese in an English-language dashboard.

In [ ]:
prod_cats = set(products['product_category_name'].dropna().unique())
translated = set(category_translation['product_category_name'].dropna().unique())

missing = sorted(prod_cats - translated)
print(f"Distinct product categories       : {len(prod_cats):,}")
print(f"Categories with English translation: {len(translated):,}")
print(f"Categories MISSING a translation   : {len(missing):,}")
if missing:
    print("\nMissing categories:")
    for c in missing:
        print(f"  - {c}")

**Finding.** A small number of Portuguese category names have no English mapping (historically `pc_gamer`, `portateis_cozinha_e_preparadores_de_alimentos`).  
**Impact.** A naive join produces null English categories for these products, which then either render as `null`/Portuguese on dashboards or get dropped by filters.  
**Action taken.** In §2 we already coalesced missing English names back to the Portuguese original as a defensive fallback. Recommend maintaining a manual translation supplement in the warehouse's `dim_category` for these stragglers.

## 6. Key Findings Summary

**Executive summary — EDA & Data Quality**

- **Scale & shape:** 9 tables, ~99k orders over ~25 months (late-2016 → late-2018). Revenue and order volume grew steadily through 2017 with a sharp Nov–Dec holiday peak — the baseline for all downstream cohort/retention work.
- **Money is log-normal, not normal:** `payment_value`, `price`, and `freight_value` are all heavily right-skewed; ~4–8% of rows are IQR outliers but they are *legitimate large orders carrying real revenue* — keep them, log-transform before modelling.
- **Reviews are J-shaped:** ~5★ dominates but 1★ is the second-largest bar — dissatisfaction is vocal. `review_score` is only weakly tied to order value, so satisfaction drivers are operational (delivery, accuracy), not price.
- **Concentration risk:** São Paulo state alone is >40% of customers; geographic KPIs must be normalised per-region or SP will swamp every aggregate.
- **Nulls are semantic, not missing:** `order_delivered_customer_date` nulls map exactly to non-`delivered` statuses, and ~58% null review-comments reflect satisfied-buyer default behaviour. Neither should be imputed.
- **Primary keys are clean:** `order_id`, `customer_id`, and the `(order_id, order_item_id)` composite all pass uniqueness — safe join keys.
- **Two reconcile-before-load actions:** (a) exclude the few impossible delivery times (`delivery_days < 0` or `> 90`) from delivery SLAs; (b) treat `payment_value` as authoritative order revenue and reserve `order_items` for product attribution, since the two reconcile within 10% for ~all orders.
- **Dimension prep needed:** aggregate geolocation to one row per zip prefix (~50× reduction) before loading `dim_geolocation`, and supplement the missing English category translations.

**Persisted artefact.** Below we save a cleaned subset — delivered orders with a sane delivery window — to `data/processed/` for reuse by downstream notebooks.

In [ ]:
# Persist a cleaned subset: delivered orders with a physically plausible delivery window.
# This is the safe denominator for any delivery-time / SLA analysis downstream.
orders_valid_delivery = delivered[
    (delivered['delivery_days'] >= 0) & (delivered['delivery_days'] <= 90)
].copy()

out_path = save_processed(orders_valid_delivery, 'orders_valid_delivery')
print(f"Saved {len(orders_valid_delivery):,} of {len(delivered):,} delivered orders "
      f"({len(orders_valid_delivery) / len(delivered) * 100:.2f}% kept).")
print(f"  -> {out_path}")